# Fixed-Parameter SARIMAX Test Evaluator

This notebook reproduces the **important transformations** from the original training pipeline and evaluates the fixed **SARIMAX(2,0,1)** model on a held-out test set.

## What this notebook does
- uses the **training set only** to learn the transformation state
- applies the same transformation logic used before
- fits **SARIMAX(2,0,1)** with `treasury_lag_1` as the exogenous regressor
- makes **test-set forecasts**
- reconstructs predictions back to the **original target scale**
- reports **RMSE** and **MAE** on the test set when actual target values are present


## Imports and optional package installation

Run this cell first. It installs missing packages only if needed.


In [1]:
import importlib
import json
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "statsmodels": "statsmodels",
}

for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

import numpy as np
import pandas as pd
from scipy.special import inv_boxcox
from scipy.stats import boxcox
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

## File paths and configuration

Update the paths below to match your local files before running.


In [2]:
# =========================
# User configuration
# =========================
TRAIN_FILE = Path("../Data/train.csv")
TEST_FILE = Path("../Data/test.csv")
OUTPUT_DIR = Path("outputs/sarimax_test_eval_notebook")

DATE_COL = "Month"
TARGET_COL = "int_rate_mean"
TREASURY_COL = "Treasury_data"

# Best model chosen from your previous cross-validation outputs
SARIMAX_ORDER = (2, 0, 1)

# Treasury lag used in the original transformed model
TREASURY_LAG = 1

## Helper data structures


In [3]:
@dataclass(frozen=True)
class TransformState:
    target_boxcox_lambda: float
    target_boxcox_shift: float
    treasury_log_shift: float


@dataclass(frozen=True)
class EvaluationResult:
    n_test_rows: int
    n_forecasts: int
    sarimax_order: tuple[int, int, int]
    rmse: float | None
    mae: float | None

## Core utility functions


In [4]:
def compute_positive_shift(series: pd.Series, buffer: float = 0.0) -> float:
    minimum = float(series.min())
    return max(0.0, -minimum + buffer)


def load_dataset(path: Path, date_col: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if date_col not in df.columns:
        raise ValueError(f"Column '{date_col}' not found in {path}.")
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)
    return df


def validate_columns(df: pd.DataFrame, required_cols: list[str], df_name: str) -> None:
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing required columns: {missing}")

## Transformation functions

These match the logic from the original pipeline:
- **Box-Cox** on the target
- **second differencing** of the transformed target
- **log transform + first differencing** of `Treasury_data`
- **lag 1** for the Treasury exogenous regressor


In [5]:
def apply_train_transformations(train_df: pd.DataFrame) -> tuple[pd.DataFrame, TransformState]:
    train_ts = train_df[[DATE_COL, TARGET_COL, TREASURY_COL]].copy()
    train_ts = train_ts.set_index(DATE_COL).sort_index()

    # Target: Box-Cox then first and second differences
    target_boxcox_shift = compute_positive_shift(train_ts[TARGET_COL])
    target_input = train_ts[TARGET_COL] + target_boxcox_shift
    train_ts["target_boxcox"], target_boxcox_lambda = boxcox(target_input)
    train_ts["target_diff1"] = train_ts["target_boxcox"].diff(1)
    train_ts["target_diff2"] = train_ts["target_boxcox"].diff(1).diff(1)

    # Treasury: shift to positive if needed, then log and first difference
    treasury_log_shift = compute_positive_shift(train_ts[TREASURY_COL], buffer=0.1)
    train_ts["treasury_shifted"] = train_ts[TREASURY_COL] + treasury_log_shift
    train_ts["treasury_log"] = np.log(train_ts["treasury_shifted"])
    train_ts["treasury_log_diff"] = train_ts["treasury_log"].diff(1)

    state = TransformState(
        target_boxcox_lambda=float(target_boxcox_lambda),
        target_boxcox_shift=float(target_boxcox_shift),
        treasury_log_shift=float(treasury_log_shift),
    )
    return train_ts, state


def build_train_model_frame(train_ts: pd.DataFrame, treasury_lag: int = 1) -> pd.DataFrame:
    frame = pd.DataFrame(index=train_ts.index)
    frame["target_diff2"] = train_ts["target_diff2"]
    frame["treasury_lag_1"] = train_ts["treasury_log_diff"].shift(treasury_lag)
    frame["prev_diff1"] = train_ts["target_diff1"].shift(1)
    frame["prev_boxcox"] = train_ts["target_boxcox"].shift(1)
    return frame.dropna()


def build_test_exog(train_ts: pd.DataFrame, test_df: pd.DataFrame, state: TransformState) -> pd.DataFrame:
    test_ts = test_df[[DATE_COL, TREASURY_COL]].copy().set_index(DATE_COL).sort_index()

    combined_treasury = pd.concat(
        [
            train_ts[[TREASURY_COL]],
            test_ts[[TREASURY_COL]],
        ],
        axis=0,
    )

    combined_treasury["treasury_shifted"] = combined_treasury[TREASURY_COL] + state.treasury_log_shift
    if (combined_treasury["treasury_shifted"] <= 0).any():
        raise ValueError(
            "Treasury transformation produced non-positive values. "
            "The training-derived log shift is not sufficient for the test data."
        )

    combined_treasury["treasury_log"] = np.log(combined_treasury["treasury_shifted"])
    combined_treasury["treasury_log_diff"] = combined_treasury["treasury_log"].diff(1)
    combined_treasury["treasury_lag_1"] = combined_treasury["treasury_log_diff"].shift(TREASURY_LAG)

    test_exog = combined_treasury.loc[test_ts.index, ["treasury_lag_1"]].copy()
    if test_exog["treasury_lag_1"].isna().any():
        missing_dates = test_exog.index[test_exog["treasury_lag_1"].isna()].strftime("%Y-%m-%d").tolist()
        raise ValueError(f"Missing exogenous values for test dates: {missing_dates}")
    return test_exog

## Inversion and reconstruction

The model forecasts `target_diff2`, so we recursively reconstruct:
1. predicted Box-Cox target
2. predicted first difference
3. predicted target on the original scale


In [6]:
def invert_boxcox(value: float, state: TransformState) -> float:
    return float(inv_boxcox(value, state.target_boxcox_lambda) - state.target_boxcox_shift)


def reconstruct_recursive_predictions(
    pred_diff2: pd.Series,
    last_train_boxcox: float,
    last_train_diff1: float,
    state: TransformState,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    prev_boxcox = float(last_train_boxcox)
    prev_diff1 = float(last_train_diff1)

    for date, diff2_hat in pred_diff2.items():
        diff2_hat = float(diff2_hat)
        boxcox_hat = diff2_hat + prev_diff1 + prev_boxcox
        original_hat = invert_boxcox(boxcox_hat, state)
        diff1_hat = boxcox_hat - prev_boxcox

        rows.append(
            {
                "date": date,
                "pred_target_diff2": diff2_hat,
                "pred_target_boxcox": boxcox_hat,
                "pred_target_diff1": diff1_hat,
                "prediction": original_hat,
            }
        )

        prev_boxcox = boxcox_hat
        prev_diff1 = diff1_hat

    return pd.DataFrame(rows)

## Model fitting and evaluation


In [7]:
def fit_and_forecast(train_frame: pd.DataFrame, test_exog: pd.DataFrame) -> pd.Series:
    model = SARIMAX(
        endog=train_frame["target_diff2"],
        exog=train_frame[["treasury_lag_1"]],
        order=SARIMAX_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False)
    forecasts = fitted.forecast(steps=len(test_exog), exog=test_exog[["treasury_lag_1"]])
    forecasts.index = test_exog.index
    return forecasts


def evaluate_predictions(pred_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, EvaluationResult]:
    has_actuals = TARGET_COL in test_df.columns

    merged = pred_df.merge(
        test_df[[DATE_COL] + ([TARGET_COL] if has_actuals else [])],
        left_on="date",
        right_on=DATE_COL,
        how="left",
    ).drop(columns=[DATE_COL])

    rmse = None
    mae = None

    if has_actuals:
        merged["actual"] = merged[TARGET_COL].astype(float)
        merged["abs_error"] = (merged["actual"] - merged["prediction"]).abs()
        merged["squared_error"] = (merged["actual"] - merged["prediction"]) ** 2
        rmse = float(np.sqrt(mean_squared_error(merged["actual"], merged["prediction"])))
        mae = float(mean_absolute_error(merged["actual"], merged["prediction"]))
    else:
        merged["actual"] = np.nan
        merged["abs_error"] = np.nan
        merged["squared_error"] = np.nan

    result = EvaluationResult(
        n_test_rows=int(len(test_df)),
        n_forecasts=int(len(merged)),
        sarimax_order=SARIMAX_ORDER,
        rmse=rmse,
        mae=mae,
    )
    return merged, result

## Run the full pipeline


In [8]:
train_df = load_dataset(TRAIN_FILE, DATE_COL)
test_df = load_dataset(TEST_FILE, DATE_COL)

validate_columns(train_df, [DATE_COL, TARGET_COL, TREASURY_COL], "Train data")
validate_columns(test_df, [DATE_COL, TREASURY_COL], "Test data")

train_ts, state = apply_train_transformations(train_df)
train_frame = build_train_model_frame(train_ts, TREASURY_LAG)

if train_frame.empty:
    raise ValueError("Training modeling frame is empty after transformations. Check data length.")

test_exog = build_test_exog(train_ts, test_df, state)
pred_diff2 = fit_and_forecast(train_frame, test_exog)

last_train_boxcox = float(train_ts["target_boxcox"].iloc[-1])
last_train_diff1 = float(train_ts["target_diff1"].iloc[-1])

pred_df = reconstruct_recursive_predictions(
    pred_diff2=pred_diff2,
    last_train_boxcox=last_train_boxcox,
    last_train_diff1=last_train_diff1,
    state=state,
)

predictions_with_metrics, result = evaluate_predictions(pred_df, test_df)

print("Fixed SARIMAX evaluation complete")
print(f"Order: {result.sarimax_order}")
print(f"Number of forecasts: {result.n_forecasts}")
if result.rmse is not None and result.mae is not None:
    print(f"RMSE: {result.rmse:.6f}")
    print(f"MAE:  {result.mae:.6f}")
else:
    print("RMSE: unavailable (test file has no target column)")
    print("MAE:  unavailable (test file has no target column)")

c:\Users\sayan\anaconda3new\envs\erdos_summer_2025\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\sayan\anaconda3new\envs\erdos_summer_2025\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


Fixed SARIMAX evaluation complete
Order: (2, 0, 1)
Number of forecasts: 35
RMSE: 1.560106
MAE:  1.186652


c:\Users\sayan\anaconda3new\envs\erdos_summer_2025\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## Inspect outputs


In [9]:
predictions_with_metrics.head()

,date,pred_target_diff2,pred_target_boxcox,pred_target_diff1,prediction,int_rate_mean,actual,abs_error,squared_error
0,2016-02-01,0.001078,2.025422,0.001621,12.321268,12.547103,12.547103,0.225835,0.051002
1,2016-03-01,0.000636,2.027679,0.002257,12.364827,12.526671,12.526671,0.161844,0.026194
2,2016-04-01,-0.000246,2.029691,0.002011,12.403792,12.429358,12.429358,0.025566,0.000654
3,2016-05-01,-0.000585,2.031117,0.001426,12.431509,12.503870,12.503870,0.072361,0.005236
4,2016-06-01,0.000407,2.032949,0.001833,12.467235,12.509503,12.509503,0.042269,0.001787


## Save outputs to disk


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pred_out = predictions_with_metrics.copy()
pred_out["date"] = pd.to_datetime(pred_out["date"]).dt.strftime("%Y-%m-%d")
pred_out.to_csv(OUTPUT_DIR / "sarimax_test_predictions.csv", index=False)

with open(OUTPUT_DIR / "sarimax_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(asdict(result), f, indent=2)

with open(OUTPUT_DIR / "transform_state.json", "w", encoding="utf-8") as f:
    json.dump(asdict(state), f, indent=2)

print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")

## Useful code snippets

### 1. Plot actual vs predicted


In [ ]:
import matplotlib.pyplot as plt

plot_df = predictions_with_metrics.copy()
plot_df["date"] = pd.to_datetime(plot_df["date"])

plt.figure(figsize=(10, 5))
plt.plot(plot_df["date"], plot_df["prediction"], label="Prediction")
if plot_df["actual"].notna().any():
    plt.plot(plot_df["date"], plot_df["actual"], label="Actual")
plt.xlabel("Date")
plt.ylabel(TARGET_COL)
plt.title("SARIMAX Test Predictions")
plt.legend()
plt.tight_layout()
plt.show()

### 2. Show only the error columns


In [ ]:
error_view = predictions_with_metrics[["date", "prediction", "actual", "abs_error", "squared_error"]].copy()
error_view.head(10)

### 3. Reuse just the final metrics


In [ ]:
metrics_summary = {
    "order": result.sarimax_order,
    "n_forecasts": result.n_forecasts,
    "rmse": result.rmse,
    "mae": result.mae,
}
metrics_summary